<a href="https://colab.research.google.com/github/SmallMGarden/scientist_girl_in_green/blob/main/%D0%98%D1%82%D0%BE%D0%B3%D0%BE%D0%B2%D1%8B%D0%B9_%D0%BF%D1%80%D0%BE%D0%B5%D0%BA%D1%82_%D0%9A%D0%BE%D1%87%D0%BA%D1%83%D1%80%D0%BE%D0%B2%D0%B0_%D0%A2%D0%BE%D0%BB%D1%81%D1%82%D0%B8%D0%BA%D0%BE%D0%B2%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U datasets langchain langchain-community langchain-huggingface langchain-text-splitters langchain-groq faiss-cpu sentence-transformers


In [ ]:
import os
import re

from datasets import load_dataset

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

In [ ]:
#Загрузка датасета и создание документов
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train[:120]")

documents = []

for i, item in enumerate(dataset):# Берём заголовок и аннотацию статьи
    title = str(item.get("title", "") or "")
    abstract = str(item.get("abstract", "") or "")

    title = re.sub(r"\s+", " ", title).strip()
    abstract = re.sub(r"\s+", " ", abstract).strip()

    full_text = f"Title: {title}\n\nAbstract: {abstract}"

    # Создаём документ LangChain с метаданными
    doc = Document(
        page_content=full_text,
        metadata={
            "id": i,
            "source": "CShorten/ML-ArXiv-Papers",
            "title": title
        }
    )
    documents.append(doc)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
#Чанкинг, фильтрация и добавление заголовка
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100, # перекрытие между чанками
    separators=["\n\n", "\n", " ", ""] # порядок разбиения
)

chunked_documents = text_splitter.split_documents(documents)

# Убираем слишком короткие чанки
filtered_chunks = [
    chunk for chunk in chunked_documents
    if len(chunk.page_content.strip()) >= 100
]

chunks_with_title = []

for chunk in filtered_chunks:
    title = chunk.metadata.get("title", "").strip()
    content = chunk.page_content.strip()

    if not content.startswith("Title:"):
        content = f"Title: {title}\n\n{content}"

    chunks_with_title.append(
        Document(
            page_content=content,
            metadata=chunk.metadata
        )
    )


In [ ]:
print("Количество чанков после разбиения:", len(chunked_documents))# Показываем общее количество чанков на каждом этапе
print("Количество чанков после фильтрации:", len(filtered_chunks))
print("Количество чанков после добавления заголовков:", len(chunks_with_title))


for i in range(3):# Показываем несколько примеров итоговых чанков
    print(f"\n--- Чанк {i+1} ---")
    print("Метаданные:")
    print(chunks_with_title[i].metadata)
    print("\nДлина чанка:")
    print(len(chunks_with_title[i].page_content))
    print("\nТекст чанка:")
    print(chunks_with_title[i].page_content[:700])

chunk_lengths = [len(doc.page_content) for doc in chunks_with_title]
# Считаем длины чанков

print("\nМинимальная длина чанка:", min(chunk_lengths))
print("Максимальная длина чанка:", max(chunk_lengths))
print("Средняя длина чанка:", sum(chunk_lengths) / len(chunk_lengths))

Количество чанков после разбиения: 429
Количество чанков после фильтрации: 324
Количество чанков после добавления заголовков: 324

--- Чанк 1 ---
Метаданные:
{'id': 0, 'source': 'CShorten/ML-ArXiv-Papers', 'title': 'Learning from compressed observations'}

Длина чанка:
544

Текст чанка:
Title: Learning from compressed observations

Abstract: The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the

--- Чанк 2 ---
Метаданные:
{'id': 0, 'source': 'CShorten/ML-ArXiv-Papers', 'title': 'Learning from compressed observations'}

Длина чанка:
539

Текст чанка:
Title: Le

Для чанкинга был выбран RecursiveCharacterTextSplitter, так как он подходит для обычных текстовых научных документов и делит текст по естественным границам, а не случайным образом.

Размер чанка был задан как 500, а перекрытие — 100, то есть около 20%, чтобы сохранять связность текста между соседними фрагментами.

После первичного разбиения было получено 429 чанков, что показывает, что тексты действительно были разбиты на части, а не остались целыми документами.

После фильтрации слишком коротких фрагментов осталось 324 чанка, то есть были удалены малоинформативные куски, например слишком короткие заголовки или обрывки.

После добавления заголовков количество чанков осталось 324, значит этот этап не создавал новых лишних фрагментов, а только улучшал содержимое уже существующих чанков.

По примерам видно, что чанки содержат связный и осмысленный текст, без явных артефактов вроде оборванных слов или полностью бессодержательных фрагментов.
Минимальная длина чанка составила 100 символов, максимальная — 633, а средняя — около 484, что можно считать нормальным результатом для выбранных параметров чанкинга.

Итоговое количество чанков выглядит разумным: это не один большой чанк на документ и не чрезмерно большое число мелких фрагментов на каждую статью.

In [ ]:
#Эмбеддинги, FAISS и retriever
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = FAISS.from_documents(
    documents=chunks_with_title,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#LLM, prompt и RAG-цепочка
os.environ["GROQ_API_KEY"] = "Вставьте сюда свой GROQ API KEY"

# Подключаем LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_retries=2
)

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant for answering questions about scientific papers.

Use only the provided context to answer the question.
If the answer is not contained in the context, say clearly:
"I could not find the answer in the provided context."

Context:
{context}

Question:
{question}

Answer:
""")

# Функция объединения чанков в один контекст
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Создаем RAG
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)


In [ ]:
#Финальное тестирование
# Три вопроса разных типов
test_questions = [
    ("Фактический вопрос", "What is discussed in the paper titled Learning from compressed observations?"),
    ("Обобщающий вопрос", "What machine learning problems are discussed in the retrieved papers?"),
    ("Уточняющий вопрос", "How is compression related to learning in the paper Learning from compressed observations?")
]

for question_type, question in test_questions:
    print("\n" + "=" * 100)
    print(question_type)
    print("Вопрос:")
    print(question)

    response = rag_chain.invoke(question)
    used_chunks = retriever.invoke(question)

    print("\nОтвет модели:")
    print(response.content)

    print("\nИспользованные чанки:")
    for i, doc in enumerate(used_chunks, start=1):
        print(f"\n--- Чанк {i} ---")
        print("Метаданные:")
        print(doc.metadata)
        print("\nТекст чанка:")
        print(doc.page_content[:600])



Фактический вопрос
Вопрос:
What is discussed in the paper titled Learning from compressed observations?

Ответ модели:
The paper titled "Learning from compressed observations" discusses the problem of statistical learning in a setting where one has perfect observation of the X-part of the sample, but the Y-part has to be communicated at a finite bit rate. The encoding of the Y-values is allowed to depend on the X-values. The paper provides an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. It also illustrates the ideas on the example of nonparametric regression in Gaussian noise.

Использованные чанки:

--- Чанк 1 ---
Метаданные:
{'id': 0, 'source': 'CShorten/ML-ArXiv-Papers', 'title': 'Learning from compressed observations'}

Текст чанка:
Title: Learning from compressed observations

Abstract: The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related

In [ ]:
#тест на отсутствие ответа
question_no_answer = "How many planets are there in the universe?"
response_no_answer = rag_chain.invoke(question_no_answer)

print("Вопрос:")
print(question_no_answer)
print("\nОтвет модели:")
print(response_no_answer.content)

Вопрос:
How many planets are there in the universe?

Ответ модели:
I could not find the answer in the provided context.


Для проверки системы были заданы три разнотипных вопроса: фактический, обобщающий и уточняющий.

---

На фактический вопрос система дала корректный ответ по статье Learning from compressed observations и верно указала, что в работе рассматривается задача статистического обучения при ограниченной передаче Y-части выборки с конечной битовой скоростью.
Для ответа на фактический вопрос основными оказались чанки 1–3 с id = 0 и заголовком Learning from compressed observations, так как именно в них содержатся постановка задачи, ограничение на передачу Y и формулировка основного результата статьи.
Чанки 4–5 из статьи Compressed Counting тоже попали в выдачу, но использовались скорее как тематически близкие по слову compressed, а не как основные источники ответа.


---


На обобщающий вопрос система выдала осмысленный обобщённый ответ и перечислила несколько направлений задач машинного обучения: statistical learning of arbitrary computable classifiers, SVM classification with indefinite kernels, multitask learning и singular statistical estimation.
Для ответа на обобщающий вопрос retriever использовал чанки из разных статей, поэтому ответ получился не по одному документу, а по нескольким работам сразу, что логично для вопроса такого типа.
На обобщающем вопросе система сработала осмысленно, но такой тип запроса по своей природе даёт более широкий и менее точный ответ, потому что поиск собирает информацию сразу из нескольких статей.

---


На уточняющий вопрос система корректно объяснила, как именно сжатие связано с обучением в статье Learning from compressed observations: Y-часть выборки передаётся с конечной битовой скоростью, а кодирование Y может зависеть от X.
Для ответа на уточняющий вопрос снова основными были чанки 1–3 статьи Learning from compressed observations, потому что именно они напрямую раскрывают связь между ограниченной передачей информации и качеством обучения.
Чанки 4–5 из Compressed Counting и здесь попали в выдачу как тематически близкие, но по смыслу были менее релевантны, чем чанки основной статьи.
В целом, система показала хорошее качество работы на фактическом и уточняющем вопросах, так как в обоих случаях retriever нашёл правильную статью и модель ответила по её содержанию.


---


Дополнительно была проверена ситуация, когда ответ отсутствует в контексте: на вопрос, не связанный с содержанием корпуса, система вернула сообщение I could not find the answer in the provided context., то есть корректно не стала придумывать информацию.

---


Полученные результаты показывают, что RAG-система умеет находить релевантные фрагменты, использовать их как контекст для LLM и формировать ответы на их основе.
При этом видно, что в top-k выдачу иногда попадают тематически близкие, но не полностью релевантные чанки, например из статьи Compressed Counting, что можно объяснить близостью терминов и ограничениями выбранной эмбеддинг-модели.
Мы делаем вывод, что система работает корректно, лучше всего отвечает на точно сформулированные вопросы по конкретной статье, а качество retrieval зависит от формулировки запроса, состава чанков и семантической близости терминов в корпусе.